ANIMA usw. entfernt, nicht Teil der Aufgabe und zu viel  
Jahressaisonalität als Ergänzung hinzugefügt  
STL umgeschrieben, da wir jetzt einfach lin Trend benutzen, kann man besser formulieren  
ACF Erklärung  
lin reg kompakter gefasst  
Erkenntnisse hinzugeschrieben, motto: why use many word when few do trick? ggf. ergänzen  
  
TODO:  
Jahressaisonalität bei manueller Zerlegung rausrechnen? -> Aufgabe 3  
Vergleich Tarife Erkenntnisse  
Diesen Text entfernen

In [ ]:
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

---

## Vorverarbeitung der Daten

In [ ]:
df = pd.read_csv("data.csv")
df["date"] = pd.to_datetime(df["DateTime"])

# Tarife werden vorerst zusammengelegt, "date" wird dabei als Index übernommen
df = df.groupby("date").agg({"KWH": "sum", "meters": "sum"})

# Normalisierung: Stromverbrauch eines Verbrauchers für jeden Zeitpunkt
df["kwh"] = df["KWH"] / df["meters"]

df = df[["KWH", "kwh", "meters"]]
df

In [ ]:
# Erste Visualisierung des Datensatzes
series = df["kwh"].sort_index()

_, ax = plt.subplots(figsize=(15, 6))
series.plot(ax=ax, color="#6B001E")
ax.set(
    title="Stromverbrauch eines Verbrauchers",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.margins(x=0)

plt.tight_layout()
plt.show()

---

## Datenbereinigung

Im Datensatz sind fehlerhaft Datenpunkte enthalten, bei denen kein Verbraucher geschaltet ist und die Messungen außerhalb dem 30-Minuten-Taktes liegen.

In [ ]:
# Prüfung auf Datenpunkte mit keinem Verbrauch
display(df.loc[(df["KWH"] == 0)])

# Falls die Prüfungen nichts zurückliefern, können die fehlerhaften Datenpunkte entfernt werden
# Prüfung auf korrekte Datenpunkte mit keinem Verbrauch
display(df.loc[(df["KWH"] == 0) & ((df.index.minute % 30 == 0) & (df.index.second == 0))])
# Prüfung auf korrekte Datenpunkte außerhalb des 30-Minuten-Taktes
display(df.loc[~(df["KWH"] == 0) & ~((df.index.minute % 30 == 0) & (df.index.second == 0))])

# Durch .asfreq() fallen die fehlerhaften Datenpunkte weg
df = df.sort_index().asfreq("30min")
df["kwh"].isna().sum()

---

## Untersuchung der Zeitreihe auf Trend, Saisonalität und Residuum

In [ ]:
def analyse_trend(df: pd.DataFrame) -> pd.DataFrame:
    # Berechnung des linearen Trends der Zeitreihe
    x = mdates.date2num(df.index)
    p = np.poly1d(np.polyfit(x, df["kwh"], 1))
    df["kwh_trend"] = p(x)

    # Der Stromverbrauch bereinigt vom Trend
    df["kwh_trendbereinigt"] = df["kwh"] - df["kwh_trend"]

    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(df.index, df["kwh"], color="#6B001E", alpha=0.3, label="Verbrauch")
    ax.plot(df.index, df["kwh_trendbereinigt"], color="#F3BE2B", alpha=0.9, label="Verbrauch (Trendbereinigt)")
    ax.plot(df.index, df["kwh_trend"], color="#000000", alpha=1, linewidth=2, label="Trend")
    ax.set(
        title="Stromverbrauch eines Verbrauchers",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()

    return df


df = analyse_trend(df)

<font color="orange">
Erkenntnisse: Es ist ein schwach fallender Trend zu erkennen - Stromverbrauch nimmt über die Zeit ab (auch, wenn man als Zeitraum volle Jahre nimmt, also vom 01.01.2012 bis 01.01.2014).
</font>

In [ ]:
def analyse_seasonality(df: pd.DataFrame) -> pd.DataFrame:
    df["focus"] = df.index.hour * 2 + (df.index.minute // 30)  # Fokus auf einen Tag setzen

    # Der Mittelwert aller 30-Minuten Intervalle eines Tages
    seasonal = df.groupby("focus")["kwh_trendbereinigt"].mean()

    _, ax = plt.subplots(figsize=(15, 6))

    # Verbrauch aller Tage zur Veranschaulichung
    for i, (date, group) in enumerate(df.groupby(df.index.date)):
        if i == 0:
            ax.plot(group["focus"], group["kwh_trendbereinigt"], color="gray", alpha=0.02, label="Alle Tage")
        else:
            ax.plot(group["focus"], group["kwh_trendbereinigt"], color="gray", alpha=0.02)

    seasonal.plot(ax=ax, color="#031E81", alpha=1, label="Durchschnittsverbrauch", linewidth=2)
    ax.set(
        title="Saisonalität (Betrachtung: Tag)",
        xlabel="Uhrzeit",
        ylabel="Kilowattstunde",
    )
    ax.set_xticks(range(0, 48, 6))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
    ax.grid(axis="both", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()

    # .transform berechnet den Mittelwert pro Gruppe und weist ihn jedem Datenpunkt der Gruppe zu
    df["kwh_trendbereinigt_saison"] = df.groupby("focus")["kwh_trendbereinigt"].transform("mean")
    return df.drop(columns=["focus"])


df = analyse_seasonality(df)

<font color="orange">
Erkenntnisse: Wie erwartet fällt der Stromverbrauch nachts sehr stark, nimmt im Laufe des Mittags zu und erreicht seinen Höhepunkt abends, wenn jeder wieder zu Hause ist.
</font>

In [ ]:
# Ergänzend
def analyse_seasonality_year(df: pd.DataFrame) -> None:
    df_year = df.copy()

    # Fokus auf ein Jahr setzen
    df_year["focus"] = df_year.index.month

    seasonal = df_year.groupby("focus")["kwh_trendbereinigt"].mean()

    _, ax = plt.subplots(figsize=(15, 6))

    seasonal.plot(ax=ax, color="#031E81", alpha=1, label="Durchschnittsverbrauch", linewidth=2)
    ax.set(
        title="Saisonalität (Betrachtung: Jahr)",
        xlabel="Monat",
        ylabel="Kilowattstunde",
    )
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(pd.Series(range(1, 13)).map(lambda x: pd.to_datetime(x, format="%m").strftime("%b")))
    ax.grid(axis="both", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()


analyse_seasonality_year(df)

<font color="orange">
Erkenntnisse: Im den Wintermonaten wird deutlich mehr Strom verbraucht als im Sommer.
</font>

In [ ]:
def analyse_cleaned(df: pd.DataFrame) -> pd.DataFrame:
    _, ax = plt.subplots(figsize=(15, 6))

    # Berechnung der bereinigte Zeitreihe bzw. dem Residuum/Rest
    df["kwh_bereinigt"] = df["kwh_trendbereinigt"] - df["kwh_trendbereinigt_saison"]

    ax.plot(df.index, df["kwh"], color="#6B001E", alpha=0.3, label="Verbrauch")
    ax.plot(df.index, df["kwh_trendbereinigt"], color="#F3BE2B", alpha=0.5, label="Verbrauch (Trendbereinigt)")
    ax.plot(df.index, df["kwh_bereinigt"], color="#9B0000", alpha=0.9, label="Verbrauch (Bereinigt/Residuum)")
    ax.plot(df.index, df["kwh_trend"], color="#000000FF", alpha=0.7, label="Trend")
    ax.plot(df.index, df["kwh_trendbereinigt_saison"], color="#D386FF", alpha=0.7, label="Saisonalität")
    ax.set(
        title="Stromverbrauch eines Verbrauchers (Fokus auf Zeitraum)",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    # Begrenzung auf Zeitraum zur besseren Lesbarkeit
    ax.set_xlim(pd.Series(df.index).quantile(0.46), pd.Series(df.index).quantile(0.52))
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()

    # Getrennte Plots
    _, ax = plt.subplots(4, 1, figsize=(15, 6), sharex=True)

    for p in ax:
        p.set_xlim(pd.Series(df.index).quantile(0.46), pd.Series(df.index).quantile(0.52))

    ax[0].plot(df["kwh"], color="#6B001E")
    ax[0].set_title("Stromverbrauch")

    ax[1].plot(df["kwh_trend"], color="#000000FF")
    ax[1].set_title("Trend")

    ax[2].plot(df["kwh_trendbereinigt_saison"], color="#D386FF")
    ax[2].set_title("Saisonalität")

    ax[3].plot(df["kwh_bereinigt"], color="#9B0000")
    ax[3].set_title("Residuum")

    plt.tight_layout()
    plt.show()

    # Residuum der gesamten Zeitreihe
    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(df.index, df["kwh_bereinigt"], color="#9B0000", alpha=0.9, label="Verbrauch (Bereinigt/Residuum)")
    ax.set(
        title="Stromverbrauch eines Verbrauchers",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()
    return df


df = analyse_cleaned(df)

---

## Verwendung von STL

Unsere manuelle Zerlegung basiert auf einem einfachen additiven Modell, welches einige Ungenauigkeiten aufweist: Ein linearer Trend ist unzureichend und eine einmalige Mittelung erfasst keine Schwankungen in der Saisonalität über die Zeit.

STL verwendet einen iterativen Prozess, der Trend und Saisonalität wechselseitig verfeinert. Die Loess-Glättung ist dabei robuster gegenüber Ausreißern, und die Saisonalitätskomponente kann sich über den Beobachtungszeitraum hinweg verändern. Das Residuum ist dadurch sauberer isoliert und repräsentiert tatsächlich nur den zufälligen Rest.

In [ ]:
from statsmodels.tsa.seasonal import STL


def plot_STL(df: pd.DataFrame) -> pd.DataFrame:
    decomposition = STL(
        df["kwh"],
        period=48,
    )
    decomposition = decomposition.fit()

    _, ax = plt.subplots(4, 1, figsize=(15, 6), sharex=True)

    for p in ax:
        p.set_xlim(pd.Series(df.index).quantile(0.46), pd.Series(df.index).quantile(0.52))

    ax[0].plot(df["kwh"], color="#6B001E")
    ax[0].set_title("Stromverbrauch")

    ax[1].plot(decomposition.trend, color="#000000FF")
    ax[1].set_title("Trend")

    ax[2].plot(decomposition.seasonal, color="#D386FF")
    ax[2].set_title("Saisonalität")

    ax[3].plot(decomposition.resid, color="#9B0000")
    ax[3].set_title("Residuum")

    plt.tight_layout()
    plt.show()

    # Residuum der gesamten Zeitreihe
    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(df.index, decomposition.resid, color="#9B0000", alpha=0.9, label="Verbrauch (Bereinigt/Residuum)")
    ax.set(
        title="Stromverbrauch eines Verbrauchers",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()

    return decomposition


decomposition = plot_STL(df)

In [ ]:
# Im Folgendem verwenden wir die berechneten Werte der STL-Methode
df["kwh_trend"] = decomposition.trend
df["kwh_trendbereinigt"] = df["kwh"] - df["kwh_trend"]
df["kwh_trendbereinigt_saison"] = decomposition.seasonal
df["kwh_bereinigt"] = decomposition.resid

---

## Autokorrelation und Partielle Autokorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

_, ax = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(df["kwh_bereinigt"], lags=48, ax=ax[0])
ax[0].set_title("Autokorrelation")

plot_pacf(df["kwh_bereinigt"], lags=48, ax=ax[1])
ax[1].set_title("Partielle Autokorrelation")

plt.tight_layout()
plt.show()

<font color="orange">
Erkenntnisse: Beim Rest der Reihe (Residuum) lässt sich weiterhin eine leichte Saisonalität (Jahrerythmus) erkennen, was das schwache Ergebnis der Autokorrelation erklären würde. Die Zeitreihe ist also nicht ganz stationär.
</font>

---

## Vorhersage mit Linearer Regression

Zielvariable hier ist zunächst der tatsächliche Verbrauch 6 Stunden in der Zukunft.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

In [ ]:
df_model = df.copy()

lags = []
for lag in range(1, 13):
    col_name = f"kwh_bereinigt_t-{lag}"
    df_model[col_name] = df_model["kwh_bereinigt"].shift(lag)
    lags.append(col_name)

df_model["kwh_target"] = df_model["kwh"].shift(-12)  # Zielwert: Vorherzusagener Wert 6 Stunden in der Zukunft
df_model = df_model.dropna()
df_model[["kwh", "kwh_bereinigt", "kwh_bereinigt_t-1", "kwh_bereinigt_t-6", "kwh_bereinigt_t-12", "kwh_target"]].head(5)

In [ ]:
# Aufteilung der Trainings- und Testdaten
split = int(len(df_model) * 0.8)
train = df_model.iloc[:split]
test = df_model.iloc[split:]

# Trainieren des Models
features_a = lags + ["kwh_trend", "kwh_trendbereinigt_saison"]
X_train_a = train[features_a]
y_train_a = train["kwh_target"]  # Um 6 Stunden verschobenen Wert nehmen, nicht den Original Wert
model_a = LinearRegression()
model_a.fit(X_train_a, y_train_a)

# Vorhersage auf Testdaten
X_test_a = test[features_a]
y_pred_a = model_a.predict(X_test_a)

In [ ]:
def plot_pred(test: pd.DataFrame, pred: np.ndarray, col: str) -> None:
    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(test.index, test[col], color="#6B001E", alpha=0.5, label="Tatsächlicher Verbrauch")
    ax.plot(test.index, pred, color="#0029BC", alpha=1, label="Prognose")

    ax.set(
        title="Prognose 6 Stunden in die Zukunft",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.set_xlim(pd.Series(test.index).quantile(0.0), pd.Series(test.index).quantile(0.1))
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()


plot_pred(test, y_pred_a, "kwh_target")

In [ ]:
def show_metrics(y_test: np.ndarray, pred: np.ndarray, model: LinearRegression) -> None:
    print(
        f"Regression Intercept: {model.intercept_}\n"
        f"Regression Coefficients: {model.coef_}\n"
        f"R²:\t{r2_score(y_test, pred): .6f}\n"
        f"MSE:\t{mean_squared_error(y_test, pred): .6f}\n"
        f"RMSE:\t{root_mean_squared_error(y_test, pred): .6f}\n"
        f"MAE:\t{mean_absolute_error(y_test, pred): .6f}\n"
    )


show_metrics(test["kwh_target"], y_pred_a, model_a)

<font color="orange">
Erkenntnisse: Schlecht.
</font>

---

Als Zielvariable wird nun das Residuum 6 Stunden in der Zukunft verwendet.

Für die eigentliche Vorhersage werden der zuvor berechnete Trend und Saisonalität wieder einberechnet. 

In [ ]:
df_model = df.copy()
lags = []
for lag in range(1, 13):
    col_name = f"kwh_bereinigt_t-{lag}"
    df_model[col_name] = df_model["kwh_bereinigt"].shift(lag)
    lags.append(col_name)

df_model["kwh_trend_t12"] = df_model["kwh_trend"].shift(-12)  # Shift von Trend um mit der Vorhersage übereinzustimmen
df_model["kwh_saison_t12"] = df_model["kwh_trendbereinigt_saison"].shift(
    -12
)  # Shift der Saisonalität um mit der Vorhersage übereinzustimmen
df_model["kwh_target"] = df_model["kwh_bereinigt"].shift(-12)  # Zielwert: Vorherzusagener Wert 6 Stunden in der Zukunft
df_model["kwh_target_final"] = df_model["kwh"].shift(-12)  # Ursprünglichen Daten für den Metrik Vergleich
df_model = df_model.dropna()

# Aufteilung der Trainings- und Testdaten
split = int(len(df_model) * 0.8)
train = df_model.iloc[:split]
test = df_model.iloc[split:]

# Trainieren des Models
features_b = lags  # Hier ohne Trend und Saisonalität da diese nicht gelernt werden soll
X_train_b = train[features_b]
y_train_b = train["kwh_target"]  # Um 6 Stunden verschobenen Wert nehmen, nicht den Original Wert.
model_b = LinearRegression()
_ = model_b.fit(X_train_b, y_train_b)

In [ ]:
# Vorhersage der Testdaten
X_test_b = test[features_b]
y_pred_residuum = model_b.predict(X_test_b)  # Nur das Residuum wird hier vorhergesagt

# Trend und Saisonalität bei t+12 liegen durch .shift(-12) bereits im test-Index vor
y_pred_b = y_pred_residuum + test["kwh_trend_t12"] + test["kwh_saison_t12"]

In [ ]:
plot_pred(test, y_pred_b, "kwh_target_final")

In [ ]:
show_metrics(test["kwh_target_final"], y_pred_b, model_b)

<font color="orange">
Erkenntnisse: Besser.
</font>

In [ ]:
# Letzter bekannter Datenpunkt für Zukunftsprognose
last_row = df_model.iloc[[-1]]
X_future_a = last_row[features_a]
X_future_b = last_row[features_b]

# Zukunftsprognose Option A
future_pred_a = model_a.predict(X_future_a)[0]
future_time = df_model.index[-1] + pd.Timedelta(hours=6)

# Zukunftsprognose Option B
future_residuum_b = model_b.predict(X_future_b)[0]
future_pred_b = future_residuum_b + last_row["kwh_trend_t12"].values[0] + last_row["kwh_saison_t12"].values[0]

_, ax = plt.subplots(figsize=(15, 7))

ax.plot(
    test.index,
    test["kwh_target_final"],
    color="#6B001E",
    alpha=0.3,
    linewidth=2,
    label="Tatsächlicher Verbrauch (t+12)",
)
ax.plot(test.index, y_pred_a, color="#C45E00", alpha=0.7, linewidth=3, label="Option A (direkt auf kwh)")
ax.plot(test.index, y_pred_b, color="#1A3D8F", alpha=0.7, linewidth=3, label="Option B (Residuum + Trend + Saison)")

ax.scatter(
    future_time, future_pred_a, color="#C45E00", s=80, zorder=5, label=f"Option A: +6h = {future_pred_a:.4f} kWh"
)
ax.scatter(
    future_time, future_pred_b, color="#1A3D8F", s=80, zorder=5, label=f"Option B: +6h = {future_pred_b:.4f} kWh"
)
ax.axvline(x=df_model.index[-1], color="gray", linestyle="--", alpha=1, label="Ende Testdaten")

ax.set(
    title="Vergleich von Option A und Option B",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.set_xlim(pd.Series(test.index).quantile(0.95), future_time + pd.Timedelta(hours=1))
ax.grid(axis="y", alpha=0.5)
ax.margins(x=0)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
r2_a, mse_a, rmse_a, mae_a = 0.155267, 0.005083, 0.071295, 0.057852  # Werte von Option A einsetzen
r2_b, mse_b, rmse_b, mae_b = 0.986163, 0.000083, 0.009125, 0.006429  # Werte von Option B einsetzen

# Metriken
r2_diff = (r2_b - r2_a) / abs(r2_a) * 100
rmse_diff = (rmse_a - rmse_b) / rmse_a * 100
mae_diff = (mae_a - mae_b) / mae_a * 100

print(
    f"{'Metrik':<8} {'Option A':>10} {'Option B':>10} {'Verbesserung':>14}\n"
    f"{'-' * 45}\n"
    f"{'R²':<8} {r2_a:>10.4f} {r2_b:>10.4f} {r2_diff:>+13.1f}%\n"
    f"{'RMSE':<8} {rmse_a:>10.4f} {rmse_b:>10.4f} {rmse_diff:>+13.1f}%\n"
    f"{'MAE':<8} {mae_a:>10.4f} {mae_b:>10.4f} {mae_diff:>+13.1f}%\n"
)

## Varianzen

In [ ]:
# Berechnung der Varianzen
def variances(df: pd.DataFrame):
    df_var = df.copy()

    df_var["trend"] = df_var["kwh_trend"]
    df_var["seasonal"] = df_var["kwh_trendbereinigt_saison"]
    df_var["resid"] = df_var["kwh_bereinigt"]

    variances = df_var[["kwh", "trend", "seasonal", "resid"]].var()
    variances["ryth"] = df_var["seasonal"].var()
    portions = variances / variances["kwh"]

    print(
        f"Von der Varianz der Zeitreihe werden:\n\t{(portions['trend'] * 100):2.1f}% durch den Trend erklärt"
        f"\n\t{(portions['ryth'] * 100):2.1f}% durch die Periodizität erklärt"
        f"\n\t{(portions['resid'] * 100):2.1f}% haben andere Ursachen (Residuum)"
    )


variances(df)

## Vergleich beider Tarife

In [ ]:
df = pd.read_csv("data.csv")


def prep_data(df: pd.DataFrame, rate: str) -> pd.DataFrame:
    df["date"] = pd.to_datetime(df["DateTime"])
    df = df.loc[df["rate"] == rate]
    df["kwh"] = df["KWH"] / df["meters"]
    return df[["date", "KWH", "kwh", "meters"]].set_index("date")[["kwh"]].sort_index().asfreq("30min")


def analyse(df: pd.DataFrame) -> pd.DataFrame:
    decomposition = plot_STL(analyse_seasonality(analyse_trend(df)))
    df["kwh_trend"] = decomposition.trend
    df["kwh_trendbereinigt"] = df["kwh"] - df["kwh_trend"]
    df["kwh_trendbereinigt_saison"] = decomposition.seasonal
    df["kwh_bereinigt"] = decomposition.resid
    return df


df_std = prep_data(df, "Std")
df_tou = prep_data(df, "ToU")

#### Standard Tarif

In [ ]:
variances(analyse(df_std))

#### Variabler Tarif

In [ ]:
variances(analyse(df_tou))

<font color="orange">
Erkenntnisse: TODO!
</font>